In [1]:
import torch
import glob
import os
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from pathlib import Path

from utils import functions as uf
from utils.model import DynamicMLP
from sklearn.model_selection import train_test_split


In [2]:
folder_path = './data/'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Caricamento di tutti i file CSV presenti nella cartella
csv_files = glob.glob(os.path.join(folder_path, '*c000.csv'))
df_all = pd.concat((pd.read_csv(file, sep=";") for file in csv_files), ignore_index=True)
    
random_seed = 34656
id_col = "user_id"

# Caricamento Forget set
forget_file_path = os.path.join(folder_path, 'forget_data.csv')
forget_df = pd.read_csv(forget_file_path)

# Estrazione ID da dimenticare
forget_ids = set(forget_df[id_col])

# Separazione netta tra Retain Set e Forget Set
retain_df = df_all[~df_all[id_col].isin(forget_ids)].reset_index(drop=True)
forget_df = df_all[df_all[id_col].isin(forget_ids)].reset_index(drop=True)

# Split Retain -> Train (80%) e Temp (20%)
train_df, temp_df = train_test_split(retain_df, test_size=0.20, random_state=random_seed)

# Split Temp -> Val (10%) e Test (10%)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=random_seed)

print(f"Dataset Sizes -> Train (Retain): {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)} | Forget: {len(forget_df)}")

# Salva gli ID di validazione richiesti per la submission
val_df[[id_col]].to_csv('validation_ids.csv', index=False)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\edoar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\edoar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\traitlets\config\application.py", line 1075, in launch

Dataset Sizes -> Train (Retain): 96558 | Val: 12070 | Test: 12070 | Forget: 9085


In [4]:
# 1. Estrazione delle matrici X e y per tutti i set dal dataframe corretto
X_train_raw, y_train, feature_cols, target_cols = uf.prepare_data(train_df, id_col=id_col, target_prefix='target__')
X_val_raw, y_val, _, _ = uf.prepare_data(val_df, id_col=id_col, target_prefix='target__')
X_test_raw, y_test, _, _ = uf.prepare_data(test_df, id_col=id_col, target_prefix='target__')
X_forget_raw, y_forget, _, _ = uf.prepare_data(forget_df, id_col=id_col, target_prefix='target__')

# 2. Imputazione coerente dei dati (fit su tutto df_all trasformato per coerenza con l'addestramento originale)
X_all_raw, _, _, _ = uf.prepare_data(df_all, id_col=id_col, target_prefix='target__')
imputer = SimpleImputer(strategy='median')
imputer.fit(X_all_raw)

X_train = imputer.transform(X_train_raw).astype(np.float32)
X_val = imputer.transform(X_val_raw).astype(np.float32)
X_test = imputer.transform(X_test_raw).astype(np.float32)
X_forget = imputer.transform(X_forget_raw).astype(np.float32)

# 3. Calcolo pos_weights sul Retain Train Set per la loss
pos_counts = np.sum(y_train, axis=0)
neg_counts = len(y_train) - pos_counts
pos_weights = torch.tensor(neg_counts / (pos_counts + 1e-6), dtype=torch.float32, device=device)
pos_weights = pos_weights.clamp(min=0.1, max=100.0)
print(f"pos_weights: {pos_weights}")

# 4. Caricamento Artifact del modello pre-addestrato
artifact_path = Path('data') / 'model_artifact'
payload = uf.load_pickle(artifact_path)

state_dict = payload['state_dict']
architecture = payload['architecture']
best_params = payload['best_hyperparameters']

print("\n--- Saved Metadata ---")
print("Architecture parameters:", architecture)
print("Best Hyperparameters:", best_params)

model = DynamicMLP(
    input_dim=architecture['input_dim'],
    hidden_layers=architecture['hidden_layers'],
    num_outputs=architecture['num_outputs']
).to(device)

model.load_state_dict(state_dict)
model.eval()

print("\nModel successfully reconstructed and weights loaded.")

DataFrame columns: ['ang_m_datediff_att_linea', 'ang_m_flag_mnp', 'ang_m_datediff_cessazione', 'ang_m_datediff_scad_sim', 'ang_m_cns_cre_res_per', 'ang_m_anz_linea', 'ang_m_datediff_mnp_in', 'ang_m_datediff_cam_prof', 'ang_m_datediff_mnp_out', 'ohe_ang_m_stato_linea_a', 'ohe_ang_m_stato_linea_c', 'ohe_ang_m_stato_linea_n', 'ohe_ang_m_stato_linea_s', 'ohe_ang_m_tipo_linea_fwa', 'ohe_ang_m_tipo_linea_pp', 'ohe_ang_m_tipo_linea_xt', 'ohe_ang_m_area_territoriale_altro_nd', 'ohe_ang_m_area_territoriale_c', 'ohe_ang_m_area_territoriale_ne', 'ohe_ang_m_area_territoriale_no', 'ohe_ang_m_area_territoriale_s', 'ohe_ang_m_pdv_canale_des_top_4_altro', 'ohe_ang_m_pdv_canale_des_top_4_grande_distribuzione', 'ohe_ang_m_pdv_canale_des_top_4_null', 'ohe_ang_m_pdv_canale_des_top_4_one_team', 'ohe_ang_m_pdv_canale_des_top_4_retail', 'ohe_ang_m_pdv_canale_des_top_4_tele_sales', 'ohe_ang_m_mnp_olo_prov_infrastrutturati', 'ohe_ang_m_mnp_olo_prov_mvno', 'ohe_ang_m_mnp_olo_prov_nd', 'ohe_ang_m_mnp_olo_dest_in

# Gradeint ascent method

In [5]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import copy

# 1. Creiamo i DataLoader per Forget e Retain
batch_size = 256

# Prepareshing dei PyTorch Tensor
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
forget_dataset = TensorDataset(torch.tensor(X_forget, dtype=torch.float32), torch.tensor(y_forget, dtype=torch.float32))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
forget_loader = DataLoader(forget_dataset, batch_size=batch_size, shuffle=True)

# 2. Definiamo la Loss Function con i pos_weights calcolati in precedenza
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

# 3. Prepariamo il modello per l'unlearning (lavoriamo su una copia per non sovrascrivere l'originale)
unlearned_model = copy.deepcopy(model)
unlearned_model.train()

# Optimizer (un learning rate basso è fondamentale per non rompere i pesi)
lr = 1e-4
optimizer = torch.optim.Adam(unlearned_model.parameters(), lr=lr)

# Coefficienti di bilanciamento
alpha = 1.0  # Peso per il Gradient Ascent (Forget)
beta = 1.0   # Peso per il Gradient Descent (Retain)

epochs = 1   # Spesso 1-3 epoche bastano per l'unlearning senza distruggere il modello

# Loop di Unlearning
train_iter = iter(train_loader)

for epoch in range(epochs):
    total_loss = 0.0
    for x_f, y_f in forget_loader:
        x_f, y_f = x_f.to(device), y_f.to(device)
        
        # Prendi un batch dal Retain set
        try:
            x_r, y_r = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            x_r, y_r = next(train_iter)
            
        x_r, y_r = x_r.to(device), y_r.to(device)
        
        optimizer.zero_grad()
        
        # Loss sul Forget Set
        out_f = unlearned_model(x_f)
        loss_forget = criterion(out_f, y_f)
        
        # Loss sul Retain Set
        out_r = unlearned_model(x_r)
        loss_retain = criterion(out_r, y_r)
        
        # Combined Loss: Sottraiamo la loss di forget (Gradient Ascent) e sommiamo la loss di retain (Gradient Descent)
        combined_loss = - alpha * loss_forget + beta * loss_retain
        
        combined_loss.backward()
        optimizer.step()
        
        total_loss += combined_loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} - Loss combinata: {total_loss / len(forget_loader):.4f}")

unlearned_model.eval()
print("Machine Unlearning completato!")

Epoch 1/1 - Loss combinata: 0.0469
Machine Unlearning completato!


# Submission

In [6]:
import time
import os
import shutil

# --- CONFIGURAZIONE ---
GROUP_NAME = "NoWINDtoday" # Cambia con il nome del tuo gruppo
VERSION = "V2"
SUBMISSION_DIR = f"{GROUP_NAME}_{VERSION}"

execution_time_sec = 4

# 2. CREAZIONE DELLA CARTELLA SUBMISSION
os.makedirs(SUBMISSION_DIR, exist_ok=True)

# 3. CREAZIONE DI execution_time.txt
with open(os.path.join(SUBMISSION_DIR, 'execution_time.txt'), 'w') as f:
    f.write(str(execution_time_sec))

val_ids = val_df[[id_col]].dropna().copy()
val_ids[id_col] = val_ids[id_col].astype(int)

val_ids.to_csv(
    os.path.join(SUBMISSION_DIR, 'validation_ids.csv'),
    index=False,
    header=True,
    encoding='utf-8'
)

# 5. SALVATAGGIO DEL NUOVO model_artifact
# Manteniamo la stessa struttura del payload originale aggiornando lo state_dict

state_dict_cpu = {k: v.cpu() for k, v in unlearned_model.state_dict().items()}

new_payload = {
    'state_dict': unlearned_model.state_dict(),
    'architecture': architecture,
    'best_hyperparameters': best_params,
    'model_class_source': payload['model_class_source'] if 'model_class_source' in payload else None
}

artifact_save_path = os.path.join(SUBMISSION_DIR, 'model_artifact')
with open(os.path.join(SUBMISSION_DIR, 'model_artifact'), 'wb') as f:
    uf.pickle.dump(new_payload, f)
print(f"\nCartella '{SUBMISSION_DIR}' creata con successo e pronta per il caricamento!")


Cartella 'NoWINDtoday_V2' creata con successo e pronta per il caricamento!
